# Step 7 — Visualization & GeoJSON Export

Reconstructs one flight using all three methods and exports to geojson.io.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

## Cell 1 — Setup: load model and flight list

In [1]:
import sys, os
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from pyproj import Geod
from scipy.interpolate import interp1d

# ── Must chdir to noel_part BEFORE importing reconstruction ───────────────────
NOEL_DIR = Path("../noel_part").resolve()
os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

print(f"Working dir: {os.getcwd()}")

BASE_DIR  = Path(".")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
RECON_DIR = BASE_DIR / "final_reconstructions"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

geod = Geod(ellps="WGS84")

from reconstruction import (TrajectoryBiGRU, FEATURE_COLS, TARGET_COLS,
                             SEQUENCE_LEN, reconstruct_gap_baseline,
                             reconstruct_gap_kalman, reconstruct_gap,
                             compute_features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrajectoryBiGRU(len(FEATURE_COLS), 128, 2, len(TARGET_COLS), 0.3).to(device)
model.load_state_dict(torch.load("models/BiGRU.pth", map_location=device))
model.eval()

flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

print(f"Model loaded on {device}")
print(f"Flights available: {len(flights)}")

c:\Users\marko\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Working dir: C:\Users\marko\Desktop\AeroML3\noel_part
Model loaded on cpu
Flights available: 2149


## Cell 2 — Select a specific flight

We use a known Ireland→New York transatlantic flight for a clean visualization.

In [2]:
# ── Select a flight to visualize ──────────────────────────────────────────────
# Run Cell 6 first to see a ranked list of flights.
# Paste any FLIGHT_NAME from that table here.
# Flights with high improvement_mean_pct show the clearest Kalman vs baseline gap.

FLIGHT_NAME = "20231007_a0f428_022616_033535"

# Find which step folder contains this flight
STEP_NAME = None
for s in CLEAN_DIR.iterdir():
    if s.is_dir() and (s / FLIGHT_NAME).exists():
        STEP_NAME = s.name
        break

if STEP_NAME is None:
    raise FileNotFoundError(
        f"Flight '{FLIGHT_NAME}' not found under {CLEAN_DIR}\n"
        "Run Cell 6 to get valid flight names."
    )

FLIGHT_DIR = CLEAN_DIR / STEP_NAME / FLIGHT_NAME
print(f"Flight : {FLIGHT_NAME}")
print(f"Step   : {STEP_NAME}")

# ── Load data ─────────────────────────────────────────────────────────────────
def _find(fd, names):
    return next((fd / n for n in names if (fd / n).exists()), None)

_ap  = _find(FLIGHT_DIR, ["adsc_clean.parquet",        "adsc.parquet"])
_bp  = _find(FLIGHT_DIR, ["adsb_before_clean.parquet", "adsb_before.parquet"])
_afp = _find(FLIGHT_DIR, ["adsb_after_clean.parquet",  "adsb_after.parquet"])

adsc = pd.read_parquet(str(_ap)).dropna(subset=["latitude","longitude"])
bef  = pd.read_parquet(str(_bp)).dropna(subset=["latitude","longitude"])
aft  = pd.read_parquet(str(_afp)).dropna(subset=["latitude","longitude"])

for _df in [adsc, bef, aft]:
    _df["timestamp"] = pd.to_datetime(_df["timestamp"], utc=True).dt.tz_localize(None)
    _df.sort_values("timestamp", inplace=True)
    _df.reset_index(drop=True, inplace=True)

# ── Gap boundaries ────────────────────────────────────────────────────────────
t_gap_start = bef["timestamp"].iloc[-1]
t_gap_end   = aft["timestamp"].iloc[0]
gap_min     = (t_gap_end - t_gap_start).total_seconds() / 60

bef_trim = bef[bef["timestamp"] >= t_gap_start - pd.Timedelta(minutes=30)].reset_index(drop=True)
aft_trim = aft[aft["timestamp"] <= t_gap_end   + pd.Timedelta(minutes=30)].reset_index(drop=True)
adsc_gap  = adsc[(adsc["timestamp"] >= t_gap_start) &
                 (adsc["timestamp"] <= t_gap_end)].reset_index(drop=True)

print(f"\nADS-B before : {len(bef):>4} pts  (trimmed to {len(bef_trim)})")
print(f"ADS-C        : {len(adsc):>4} pts  ({len(adsc_gap)} inside gap)")
print(f"ADS-B after  : {len(aft):>4} pts  (trimmed to {len(aft_trim)})")
print(f"Gap          : {gap_min:.1f} min")
print(f"From         : lat={bef_trim['latitude'].iloc[-1]:.2f}  lon={bef_trim['longitude'].iloc[-1]:.2f}")
print(f"To           : lat={aft_trim['latitude'].iloc[0]:.2f}  lon={aft_trim['longitude'].iloc[0]:.2f}")

Flight : 20231007_a0f428_022616_033535
Step   : step1_raw_2023-10-01_to_2023-11-01

ADS-B before :  241 pts  (trimmed to 121)
ADS-C        :  278 pts  (278 inside gap)
ADS-B after  :  365 pts  (trimmed to 121)
Gap          : 201.5 min
From         : lat=52.75  lon=-63.29
To           : lat=57.02  lon=-10.12


## Cell 3 — Reconstruct the gap with all three methods

All three methods use only ADS-B before and after — ADS-C is never seen during reconstruction.

In [3]:
# ── Import new anchored Kalman from step5 module (always reload latest) ───────
import sys as _sys, importlib
_sys.path.insert(0, str(NOEL_DIR.parent / "notebook"))
import step5_kalman_aeroml3
importlib.reload(step5_kalman_aeroml3)
from step5_kalman_aeroml3 import reconstruct_single_kalman

# ── Baseline: correct constant-speed great-circle interpolation ───────────────
def reconstruct_forward_only(before_df, after_df, dt=15.0):
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    n_steps = max(1, int(round((first_time - last_time).total_seconds() / dt)))
    lat0 = float(before_df["latitude"].iloc[-1]);  lon0 = float(before_df["longitude"].iloc[-1])
    alt0 = float(before_df["altitude"].iloc[-1])
    lat1 = float(after_df["latitude"].iloc[0]);    lon1 = float(after_df["longitude"].iloc[0])
    alt1 = float(after_df["altitude"].iloc[0])
    pts  = geod.npts(lon0, lat0, lon1, lat1, n_steps)
    lats = np.array([p[1] for p in pts])
    lons = np.array([p[0] for p in pts])
    alts = np.linspace(alt0, alt1, n_steps)
    timestamps = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    result = pd.DataFrame({"latitude": lats, "longitude": lons, "altitude": alts})
    result["timestamp"] = timestamps; result["interpolated"] = True
    return result

# ── Retime helper ─────────────────────────────────────────────────────────────
def retime_to_constant_speed(recon_df, before_df, after_df, dt=15.0):
    from scipy.interpolate import interp1d
    lats = recon_df["latitude"].values
    lons = recon_df["longitude"].values
    alts = recon_df["altitude"].values if "altitude" in recon_df.columns else np.zeros(len(recon_df))
    cum_dist = np.zeros(len(lats))
    for i in range(1, len(lats)):
        _, _, d = geod.inv(float(lons[i-1]), float(lats[i-1]), float(lons[i]), float(lats[i]))
        cum_dist[i] = cum_dist[i-1] + abs(d)
    total_dist = cum_dist[-1]
    if total_dist == 0:
        return recon_df
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    total_sec  = (first_time - last_time).total_seconds()
    n_steps    = max(1, int(round(total_sec / dt)))
    time_fracs   = np.array([dt*(i+1)/total_sec for i in range(n_steps)])
    target_fracs = np.clip(time_fracs, 0.0, 1.0)
    old_fracs    = cum_dist / total_dist
    f_la  = interp1d(old_fracs, lats, bounds_error=False, fill_value=(lats[0], lats[-1]))
    f_lo  = interp1d(old_fracs, lons, bounds_error=False, fill_value=(lons[0], lons[-1]))
    f_alt = interp1d(old_fracs, alts, bounds_error=False, fill_value=(alts[0], alts[-1]))
    new_ts = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    return pd.DataFrame({"latitude": f_la(target_fracs), "longitude": f_lo(target_fracs),
                         "altitude": f_alt(target_fracs), "timestamp": new_ts, "interpolated": True})

# ── Run all three methods ─────────────────────────────────────────────────────
print("Reconstructing gap with all three methods...")

recon_base = reconstruct_forward_only(bef_trim, aft_trim)

# Pass full bef (not bef_trim) so _get_entry_velocity can find the last real heading
# even when valid ADS-B velocity data is older than 30 min before the gap
recon_kalman_rt = reconstruct_single_kalman(bef, aft_trim, dt=15.0)

recon_bigru     = compute_features(reconstruct_gap(
    model, bef_trim, aft_trim, FEATURE_COLS, TARGET_COLS, SEQUENCE_LEN, device))
recon_bigru_rt  = retime_to_constant_speed(recon_bigru, bef_trim, aft_trim)

# ── Anchor verification ───────────────────────────────────────────────────────
from step5_kalman_aeroml3 import haversine_m as _hav
_d0 = _hav(recon_kalman_rt["latitude"].iloc[0],  recon_kalman_rt["longitude"].iloc[0],
           bef["latitude"].iloc[-1],              bef["longitude"].iloc[-1])
_d1 = _hav(recon_kalman_rt["latitude"].iloc[-1], recon_kalman_rt["longitude"].iloc[-1],
           aft_trim["latitude"].iloc[0],          aft_trim["longitude"].iloc[0])
print(f"Baseline : {len(recon_base)} steps")
print(f"Kalman   : {len(recon_kalman_rt)} steps  start_gap={_d0:.1f}m  end_gap={_d1:.1f}m")
print(f"BiGRU    : {len(recon_bigru_rt)} steps")

Reconstructing gap with all three methods...
Baseline : 806 steps
Kalman   : 807 steps  start_gap=0.0m  end_gap=0.0m
BiGRU    : 806 steps


## Cell 4 — Measure error against ADS-C ground truth

ADS-C was never used during reconstruction — it is now revealed to measure error.

In [4]:
def error_km(recon_df, truth_df):
    """
    Mean closest-point distance (km) between reconstruction and ADS-C truth.
    Uses nearest-neighbour matching so timestamp misalignment doesn't matter.
    """
    if len(truth_df) == 0 or len(recon_df) == 0:
        return float("nan")

    def haversine_m(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = lat2-lat1, lon2-lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

    errs = []
    for _, row in truth_df.iterrows():
        dists = haversine_m(
            row["latitude"], row["longitude"],
            recon_df["latitude"].values, recon_df["longitude"].values,
        )
        errs.append(dists.min())
    return np.mean(errs) / 1000

base_err   = error_km(recon_base,       adsc_gap)
kalman_err = error_km(recon_kalman_rt,  adsc_gap)
bigru_err  = error_km(recon_bigru_rt,   adsc_gap)

print(f"ADS-C waypoints used for evaluation: {len(adsc_gap)}")
print()
print(f"{'Method':<20} {'Mean error (km)':>16}")
print("-" * 38)
print(f"  {'Baseline':<18} {base_err:>14.1f} km")
print(f"  {'Kalman filter':<18} {kalman_err:>14.1f} km")
print(f"  {'BiGRU model':<18} {bigru_err:>14.1f} km")
print()

best = min([("Baseline", base_err), ("Kalman", kalman_err), ("BiGRU", bigru_err)],
           key=lambda x: x[1])
print(f"Best method: {best[0]} ({best[1]:.1f} km)")

ADS-C waypoints used for evaluation: 278

Method                Mean error (km)
--------------------------------------
  Baseline                    156.6 km
  Kalman filter                19.9 km
  BiGRU model                 223.0 km

Best method: Kalman (19.9 km)


## Cell 5 — Export GeoJSON for geojson.io

Exports all three methods + ADS-C ground truth as a single GeoJSON file.
Open at https://geojson.io

In [5]:
import json

def _line(df, lat_col, lon_col, props):
    coords = [[float(lo), float(la)] for la, lo in zip(df[lat_col], df[lon_col])
              if np.isfinite(float(la)) and np.isfinite(float(lo))]
    if len(coords) < 2:
        return None
    return {"type": "Feature", "properties": props,
            "geometry": {"type": "LineString", "coordinates": coords}}

features = []

# ADS-B boundary context (grey)
features.append(_line(bef_trim, "latitude", "longitude",
    {"label": "ADS-B before (context)",
     "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}))
features.append(_line(aft_trim, "latitude", "longitude",
    {"label": "ADS-B after (context)",
     "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}))

# ADS-C ground truth (yellow) — NOT used in reconstruction
features.append(_line(adsc_gap, "latitude", "longitude",
    {"label": f"ADS-C ground truth ({len(adsc_gap)} waypoints)",
     "stroke": "#FFC107", "stroke-width": 4, "stroke-opacity": 1.0}))

# Three independent reconstructions — none saw ADS-C
features.append(_line(recon_base, "latitude", "longitude",
    {"label": f"Baseline (great-circle) — {base_err:.1f} km",
     "stroke": "#F44336", "stroke-width": 2}))
features.append(_line(recon_kalman_rt, "latitude", "longitude",
    {"label": f"Kalman filter — {kalman_err:.1f} km",
     "stroke": "#00BCD4", "stroke-width": 3}))
features.append(_line(recon_bigru_rt, "latitude", "longitude",
    {"label": f"BiGRU model — {bigru_err:.1f} km",
     "stroke": "#4CAF50", "stroke-width": 3}))

features = [f for f in features if f]
fc  = {"type": "FeatureCollection", "features": features}
out = OUT_DIR / f"{FLIGHT_NAME}_final_comparison.geojson"
out.write_text(json.dumps(fc, indent=2), encoding="utf-8")

print(f"GeoJSON saved → {out}")
print("Open at https://geojson.io")
print()
print("Legend:")
print("  Grey   = ADS-B track (boundary context, 30 min around gap)")
print("  Yellow = ADS-C ground truth (held-out, never used in reconstruction)")
print(f"  Red    = Baseline (great-circle)   — {base_err:.1f} km error")
print(f"  Cyan   = Kalman filter              — {kalman_err:.1f} km error")
print(f"  Green  = BiGRU model                — {bigru_err:.1f} km error")

GeoJSON saved → ..\outputs\20231007_a0f428_022616_033535_final_comparison.geojson
Open at https://geojson.io

Legend:
  Grey   = ADS-B track (boundary context, 30 min around gap)
  Yellow = ADS-C ground truth (held-out, never used in reconstruction)
  Red    = Baseline (great-circle)   — 156.6 km error
  Cyan   = Kalman filter              — 19.9 km error
  Green  = BiGRU model                — 223.0 km error


## Cell 6 — (Optional) Find the best North Atlantic flight to visualize

In [6]:
import pandas as pd
from pathlib import Path

# ── Load step-5 per-flight metrics ────────────────────────────────────────────
metrics_path = Path("../artifacts/step5_kalman/per_flight_metrics_test.csv")
if not metrics_path.exists():
    print("Run 05a_step5_kalman_evaluation.ipynb first to generate metrics.")
else:
    metrics = pd.read_csv(metrics_path)
    metrics[["step_name", "flight_name"]] = metrics["segment_id"].str.split("/", n=1, expand=True)

    # Absolute improvement in km — this is what shows up visually on the map
    metrics["impr_abs_km"] = metrics["baseline_mean_error_km"] - metrics["kalman_mean_error_km"]

    # Best flights to visualize: large ABSOLUTE improvement (not just %)
    top = (metrics
           .sort_values("impr_abs_km", ascending=False)
           .head(30)
           [["flight_name", "step_name",
             "baseline_mean_error_km", "kalman_mean_error_km",
             "impr_abs_km", "improvement_mean_pct", "n_waypoints"]]
           .reset_index(drop=True))

    print("Top 30 flights — largest ABSOLUTE Kalman improvement (km)")
    print("These show the most clearly different trajectories on the map.")
    print("Copy a FLIGHT_NAME from 'flight_name' column into Cell 2.\n")
    display(top)

    print("\n── Flights where Kalman is WORSE ──")
    worst = (metrics
             .sort_values("impr_abs_km", ascending=True)
             .head(10)
             [["flight_name","step_name","baseline_mean_error_km",
               "kalman_mean_error_km","impr_abs_km"]]
             .reset_index(drop=True))
    display(worst)

Top 30 flights — largest ABSOLUTE Kalman improvement (km)
These show the most clearly different trajectories on the map.
Copy a FLIGHT_NAME from 'flight_name' column into Cell 2.



,flight_name,step_name,baseline_mean_error_km,kalman_mean_error_km,impr_abs_km,improvement_mean_pct,n_waypoints
0,20250319_738101_172336_192229,step1_raw_20250318_20250319,1600.321411,1248.779297,351.542114,21.966970,64
1,20241219_407af1_183520_203554,step1_raw_2024-12-01_to_2025-01-01,374.650726,80.985771,293.664955,78.383659,64
2,20250703_aab812_102726_123306,step1_raw_2025-07-01_to_2025-08-01,420.288177,133.697739,286.590439,68.189034,64
3,20250214_4bb0e5_211201_013458,step1_raw_20250214_20250215,660.342773,387.648224,272.694550,41.295910,64
4,20250318_06a0ad_214418_235137,step1_raw_20250318_20250319,473.308807,245.025299,228.283508,48.231411,64
5,20231015_a11bc8_011803_023833,step1_raw_2023-10-01_to_2023-11-01,344.419312,133.575256,210.844055,61.217262,64
6,20250215_40779a_010648_015132,step1_raw_20250214_20250215,314.703064,121.917702,192.785362,61.259449,64
7,20230827_7380c2_070704_083342,step1_raw_2023-08-10_to_2023-09-10,326.813324,143.273865,183.539459,56.160336,64
8,20250318_406a9e_151247_193755,step1_raw_20250318_20250319,307.085693,129.064163,178.021530,57.971287,64
9,20240429_aa92fa_123136_143713,step1_raw_2024-04-01_to_2024-05-01,224.326111,62.323303,162.002808,72.217545,64



── Flights where Kalman is WORSE ──


,flight_name,step_name,baseline_mean_error_km,kalman_mean_error_km,impr_abs_km
0,20240312_7103d6_155901_162237,step1_raw_20240312_20240313,31.878222,350.721283,-318.843061
1,20241225_4076e8_182736_194804,step1_raw_2024-12-01_to_2025-01-01,101.098022,268.318695,-167.220673
2,20231109_06a129_044822_055214,step1_raw_2023-11-01_to_2023-12-01,116.367516,277.789459,-161.421944
3,20230714_040103_100043_105611,step1_raw_20230707_20230731,75.798424,198.766937,-122.968513
4,20240824_a11bc8_075228_085728,step1_raw_2024-08-10_to_2024-09-10,123.473610,231.493652,-108.020042
5,20230710_89617e_103608_114219,step1_raw_20230707_20230731,74.379608,182.250961,-107.871353
6,20231111_ac0906_040829_051908,step1_raw_2023-11-01_to_2023-12-01,114.449905,212.083786,-97.633881
7,20231103_06a11e_042309_054539,step1_raw_2023-11-01_to_2023-12-01,164.232269,260.915283,-96.683014
8,20231019_ac2cb2_031457_045201,step1_raw_2023-10-01_to_2023-11-01,153.691971,248.561096,-94.869125
9,20250408_4d0104_133501_144715,step1_raw_20250408_20250409,65.902634,153.397842,-87.495209
